# 06 — Review Outputs

Review the complete migration from Bronze through modeling, persist a final pass/fail scorecard, and keep source-coverage and model caveats visible for human sign-off.

In [ ]:
import importlib
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import mlflow
from pyspark.sql import functions as F

cwd = Path.cwd()
repo_root = next(
    (candidate for candidate in [cwd, *cwd.parents] if (candidate / "src" / "urban_workflow").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the bundle root containing src/urban_workflow.")
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
sys.modules.pop("urban_workflow.review.operations", None)
importlib.invalidate_caches()

from urban_workflow.review.operations import (
    build_migration_review, validate_migration_review,
    write_snapshot_table,
)
print(f"Bundle root: {repo_root}")

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "urban_workflow")
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
if not all((CATALOG, SCHEMA)):
    raise ValueError("catalog and schema must be non-empty.")
tables = {
    "bronze_emvs": f"{CATALOG}.{SCHEMA}.bronze_emvs_housing",
    "bronze_population": f"{CATALOG}.{SCHEMA}.bronze_population_district_barrio",
    "bronze_indicators": f"{CATALOG}.{SCHEMA}.bronze_district_indicator_panel",
    "bronze_boundaries": f"{CATALOG}.{SCHEMA}.bronze_district_boundaries",
    "bronze_grid": f"{CATALOG}.{SCHEMA}.bronze_combined_grid_snapshot",
    "silver_districts": f"{CATALOG}.{SCHEMA}.silver_district_reference",
    "silver_emvs": f"{CATALOG}.{SCHEMA}.silver_emvs_housing",
    "silver_population": f"{CATALOG}.{SCHEMA}.silver_population",
    "silver_indicators": f"{CATALOG}.{SCHEMA}.silver_indicator_panel",
    "silver_boundaries": f"{CATALOG}.{SCHEMA}.silver_district_boundaries",
    "gold_grid": f"{CATALOG}.{SCHEMA}.gold_grid_features",
    "gold_districts": f"{CATALOG}.{SCHEMA}.gold_district_features",
    "grid_typology": f"{CATALOG}.{SCHEMA}.gold_grid_typology",
    "cluster_profiles": f"{CATALOG}.{SCHEMA}.gold_cluster_profiles",
    "district_cluster_mix": f"{CATALOG}.{SCHEMA}.gold_district_cluster_mix",
    "district_anomalies": f"{CATALOG}.{SCHEMA}.gold_district_anomalies",
}
REVIEW_TABLE = f"{CATALOG}.{SCHEMA}.gold_migration_review"
reviewed_at = datetime.now(timezone.utc)
review_run_id = (
    f"review_{reviewed_at.strftime('%Y%m%dT%H%M%SZ')}_"
    f"{uuid4().hex[:8]}"
)
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Review run: {review_run_id}")

## 1. Migration scorecard

The scorecard verifies all 16 persisted table counts, accepted grid coverage, KMeans prediction/profile reconciliation, district cluster shares, the fixed-snapshot anomaly results, explanations, and the silhouette metric stored in MLflow.

In [ ]:
cluster_profiles = spark.table(tables["cluster_profiles"])
kmeans_run_ids = [
    row["mlflow_run_id"]
    for row in cluster_profiles.select("mlflow_run_id").distinct().collect()
]
if len(kmeans_run_ids) != 1 or not kmeans_run_ids[0]:
    raise ValueError(f"Expected one KMeans MLflow run ID, found {kmeans_run_ids}.")
kmeans_mlflow_run = mlflow.get_run(kmeans_run_ids[0])
silhouette_score = float(kmeans_mlflow_run.data.metrics["silhouette_score"])

review = build_migration_review(
    spark,
    tables=tables,
    silhouette_score=silhouette_score,
    review_run_id=review_run_id,
    reviewed_at_utc=reviewed_at,
)
review_validation = validate_migration_review(review)
print(review_validation)
display(review.orderBy("check_group", "check_name"))

## 2. Human model review

Cluster numbers are identifiers rather than ordered quality scores. IsolationForest flags are relative mismatch signals across only 21 districts. The listed explanation features are the largest standardized deviations and do not establish causes.

In [ ]:
display(
    cluster_profiles.select(
        "cluster_label", "narrative_label", "cell_count",
        "share_of_eligible_cells", "dominant_land_use_class",
        "mean_building_height_mean_m",
        "mean_public_transport_stop_count",
    ).orderBy("cluster_label")
)
district_anomalies = spark.table(tables["district_anomalies"])
display(
    district_anomalies.select(
        "anomaly_rank", "district_code", "district_name",
        "anomaly_score", "anomaly_flag",
        "anomaly_top_features", "anomaly_interpretation_notes",
    ).orderBy("anomaly_rank")
)
print({
    "kmeans_mlflow_run_id": kmeans_run_ids[0],
    "silhouette_score": silhouette_score,
    "exploratory_anomaly_flags": district_anomalies.filter("anomaly_flag").count(),
})

## 3. Persist final review

The review table is a deterministic sign-off artifact for the current fixed snapshots and model configuration. A later source refresh or model-policy change must create and pass a new review run.

In [ ]:
write_snapshot_table(review, REVIEW_TABLE)
spark.sql(
    f"COMMENT ON TABLE {REVIEW_TABLE} IS "
    "'Final Bronze-to-model migration acceptance scorecard for the fixed Madrid snapshots.'"
)
persisted_review = spark.table(REVIEW_TABLE)
persisted_validation = validate_migration_review(persisted_review)
print(persisted_validation)
print(
    f"Validated {persisted_review.count()} migration review checks "
    "with zero failures."
)
dbutils.notebook.exit(
    f"SUCCESS: Validated {persisted_review.count()} migration checks "
    "with zero failures."
)

## 4. Review status

A passing scorecard completes the workflow path. The bundle-managed Lakeflow Job orchestrates the parameterized notebooks from source validation through this review checkpoint. Reconstructing the derived grid remains an optional follow-up track; a dashboard is outside the completed migration scope.